<a href="https://colab.research.google.com/github/galipaydin/pes3850/blob/main/notebooks/lab08_wearables_activity_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# PES 3850 · Lab 8 — Classify your own movement
**Artificial Intelligence Applications in Sport · Week 9**

⏱️ About 90 minutes · 🧪 This notebook creates its own data — nothing can fail to download · 📱 bring the phone recording you made as homework

> **First:** `File → Save a copy in Drive`. Cells marked ✏️ **EDIT ME** are the ones you change.


### What you will produce
An activity classifier built from accelerometer data, and the discovery — by measuring it yourself — that a model
trained on **your** movement does not work on your classmate's.

### The one idea in this lab: windowing
A sensor gives you a continuous signal at 100 Hz. Machine learning wants a table. The bridge is **windowing**:
cut the signal into 2-second slices, compute a handful of numbers per slice (mean, spread, range, energy), and
you now have an ordinary table to which every method from Weeks 4–7 applies. That is the whole trick.

> **Homework path.** If you exported a Phyphox CSV, use Block 2b to upload it. If anything went wrong with your
> recording, Block 2a generates realistic signals so you can still complete the lab.


---
## Block 1 — Setup


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GroupKFold
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
RNG = np.random.default_rng(9)
FS  = 100      # sampling rate, Hz — a phone accelerometer typically runs at 100 Hz
print('Setup complete.')


---
## Block 2a — Signals (fallback / demonstration data)

Twenty people, three activities, 60 seconds each. Every person has a slightly different cadence, amplitude and
sensor orientation — exactly what makes cross-subject generalisation hard in the real world.


In [ ]:
ACTIVITIES = ['walk', 'jog', 'jump']
BASE = {'walk': dict(f=1.9, amp=2.6), 'jog': dict(f=2.5, amp=4.2), 'jump': dict(f=1.1, amp=5.2)}

def make_person(pid, seconds=60):
    """One person's recording: their own cadence, amplitude, sensor gain and pocket orientation."""
    cad  = RNG.normal(1.0, .18)          # personal cadence
    amp  = RNG.normal(1.0, .32)          # personal amplitude
    gain = RNG.normal(1.0, .12)          # this phone's accelerometer gain
    tilt = RNG.uniform(-.9, .9, 3)       # how the phone sits in the pocket
    out = []
    for act in ACTIVITIES:
        n = seconds * FS
        t = np.arange(n) / FS
        f, a = BASE[act]['f'] * cad, BASE[act]['amp'] * amp
        if act == 'jump':
            base = a*np.sin(2*np.pi*f*t) + 0.9*a*np.sin(6*np.pi*f*t)   # impact harmonics
        else:
            base = a*np.sin(2*np.pi*f*t) + 0.35*a*np.sin(4*np.pi*f*t)
        ax_ = gain*(base + 9.81*np.sin(tilt[0]))                    + RNG.normal(0, 1.1, n)
        ay_ = gain*(.55*base*np.cos(np.pi*f*t) + 9.81*np.sin(tilt[1])) + RNG.normal(0, 1.1, n)
        az_ = gain*(9.81*np.cos(tilt[2]) + .40*base)                + RNG.normal(0, 1.2, n)
        out.append(pd.DataFrame({'person': pid, 'activity': act, 't': t,
                                 'ax': ax_, 'ay': ay_, 'az': az_}))
    return pd.concat(out, ignore_index=True)

raw = pd.concat([make_person(f'S{i:02d}') for i in range(20)], ignore_index=True)
ME = 'S00'
print(f'{len(raw):,} samples · {raw.person.nunique()} people · {FS} Hz · {ACTIVITIES}')
raw.head()


---
## Block 2b — Your own recording (optional)

Uncomment and run to upload the Phyphox CSV you exported. Phyphox names its columns something like
`Time (s)`, `Acceleration x (m/s^2)` … — rename them to `t`, `ax`, `ay`, `az` and set `activity` for each file.


In [ ]:
# from google.colab import files
# up = files.upload()                       # choose your three exported CSVs
# mine = []
# for fname, act in zip(up.keys(), ['walk','jog','jump']):
#     one = pd.read_csv(fname)
#     one.columns = ['t','ax','ay','az'][:len(one.columns)]
#     one['person'], one['activity'] = 'ME', act
#     mine.append(one)
# raw = pd.concat([pd.concat(mine, ignore_index=True), raw], ignore_index=True)
# ME = 'ME'
# print('your recording added:', (raw.person == 'ME').sum(), 'samples')


---
## Block 3 — Look at the signal before you model it

You can see the three activities with your own eyes. Establish that first — a model that cannot separate what you
can see by eye is broken, and a model that separates what you cannot see deserves suspicion.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 6), sharex=True)
for ax, act in zip(axes, ACTIVITIES):
    seg = raw[(raw.person == ME) & (raw.activity == act)].head(6*FS)
    ax.plot(seg.t, seg.ax, lw=.8, label='x', color='#8A1C2B')
    ax.plot(seg.t, seg.ay, lw=.8, label='y', color='#1B6470')
    ax.plot(seg.t, seg.az, lw=.8, label='z', color='#9A5714')
    ax.set_ylabel(f'{act}\nm/s²'); ax.grid(alpha=.3)
axes[0].legend(ncol=3, frameon=False, loc='upper right')
axes[-1].set_xlabel('Time (s)')
fig.suptitle(f'Six seconds of {ME} — three activities, raw tri-axial acceleration')
plt.tight_layout(); plt.show()


---
## ✏️ EDIT ME — windowing and features

`WINDOW_S` is the length of each slice. Two seconds is conventional for gait; try 0.5 and 5 later and see what
happens to accuracy — and think about *why*.


In [ ]:
WINDOW_S = 2.0    # ✏️ EDIT ME — window length in seconds. Try 0.5, 2.0, 5.0
OVERLAP  = 0.5    # ✏️ EDIT ME — 0.5 means each window overlaps the previous one by half

def features(seg):
    """Turn one window of signal into one row of numbers."""
    f = {}
    mag = np.sqrt(seg.ax**2 + seg.ay**2 + seg.az**2)
    for name, s in [('ax', seg.ax), ('ay', seg.ay), ('az', seg.az), ('mag', mag)]:
        f[f'{name}_mean'] = s.mean()
        f[f'{name}_sd']   = s.std()
        f[f'{name}_min']  = s.min()
        f[f'{name}_max']  = s.max()
        f[f'{name}_rng']  = s.max() - s.min()
        f[f'{name}_sma']  = np.abs(s).mean()          # signal magnitude area
    # dominant frequency of the magnitude signal — cadence, essentially
    sp = np.abs(np.fft.rfft(mag - mag.mean()))
    fr = np.fft.rfftfreq(len(mag), 1/FS)
    f['dom_freq'] = fr[np.argmax(sp)]
    return f

win, step = int(WINDOW_S*FS), int(WINDOW_S*FS*(1-OVERLAP))
rows = []
for (pid, act), g in raw.groupby(['person','activity']):
    g = g.reset_index(drop=True)
    for s in range(0, len(g) - win + 1, step):
        r = features(g.iloc[s:s+win])
        r['person'], r['activity'] = pid, act
        rows.append(r)

W = pd.DataFrame(rows)
FEATS = [c for c in W.columns if c not in ('person','activity')]
print(f'{len(W)} windows × {len(FEATS)} features  (from {len(raw):,} raw samples)')
W.head()


---
## Block 4 — Train on yourself, test on yourself


In [ ]:
mine = W[W.person == ME]
Xm, ym = mine[FEATS], mine.activity
Xtr, Xte, ytr, yte = train_test_split(Xm, ym, test_size=.3, random_state=0, stratify=ym)

clf = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1).fit(Xtr, ytr)
own = accuracy_score(yte, clf.predict(Xte))
print(f'Trained on {ME}, tested on held-out windows from {ME}:  {own:.1%}')
print('Excellent. Now find out what that number is worth.')


---
## Block 5 — Now test it on everyone else 💥


In [ ]:
others = {}
for pid, g in W.groupby('person'):
    if pid == ME: continue
    others[pid] = accuracy_score(g.activity, clf.predict(g[FEATS]))
others = pd.Series(others).sort_values()

print(f'Tested on {ME} (held-out windows) : {own:.1%}')
print(f'Tested on the other {len(others)} people  : mean {others.mean():.1%}, '
      f'worst {others.min():.1%} ({others.idxmin()}), best {others.max():.1%}')
print(f'Chance with 3 classes            : 33.3%')

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.bar(others.index, others.values,
       color=['#8A1C2B' if v < .5 else '#9A5714' if v < .85 else '#1B6470' for v in others.values])
ax.axhline(own, ls='--', color='#1B6470', label=f'tested on {ME} itself ({own:.0%})')
ax.axhline(1/3, ls=':', color='#8A8078', label='chance (33%)')
ax.set_ylabel('Accuracy'); ax.set_xlabel('Person the model was tested on')
ax.set_title(f'One model trained on {ME}, tested on everyone else')
ax.legend(frameon=False, fontsize=9); ax.set_ylim(0, 1.05); ax.grid(axis='y', alpha=.3)
plt.xticks(rotation=60, fontsize=8); plt.tight_layout(); plt.show()

worst = others.idxmin()
cm = confusion_matrix(W[W.person==worst].activity, clf.predict(W[W.person==worst][FEATS]), labels=ACTIVITIES)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay(confusion_matrix(yte, clf.predict(Xte), labels=ACTIVITIES),
                       display_labels=ACTIVITIES).plot(ax=ax[0], colorbar=False, cmap='Blues')
ax[0].set_title(f'Tested on {ME} — {own:.0%}')
ConfusionMatrixDisplay(cm, display_labels=ACTIVITIES).plot(ax=ax[1], colorbar=False, cmap='Reds')
ax[1].set_title(f'Tested on {worst} — {others.min():.0%}')
plt.tight_layout(); plt.show()


### Read that chart again

Your model is **perfect on you**. On some classmates it is also perfect. On others it is **at chance** — it has
learned nothing that transfers to them at all. And nothing in the 100% you measured on yourself told you which
classmates those would be.

The model learned *your* cadence, *your* amplitude, *your* phone's gain, and how *your* phone sat in *your* pocket.
Those are stable within a person and different between people, so a within-person test is nearly free marks.

It is the identical error as putting the same athlete in the training and test sets of an injury model. Person here,
athlete there — and the fix is the same: **group your splits by the unit you will actually generalise across.**


---
## Block 6 — Four numbers, one dataset


In [ ]:
X, y, g = W[FEATS], W.activity, W.person

naive   = cross_val_score(RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1),
                          X, y, cv=5).mean()
grouped = cross_val_score(RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1),
                          X, y, cv=GroupKFold(n_splits=5), groups=g).mean()

print(f'Trained on {ME} alone, tested on others (mean) : {others.mean():.1%}')
print(f'Pooled, random 5-fold across all {g.nunique()} people   : {naive:.1%}')
print(f'Pooled, grouped 5-fold (leave people out)     : {grouped:.1%}   <- the number to report')
print(f'Chance                                        : {1/3:.1%}')

fig, ax = plt.subplots(figsize=(7.5, 3.4))
labels = [f'Trained on {ME} only\n(tested on others)', 'Pooled, random split\n(inflated)',
          'Pooled, grouped by person\n(honest)', 'Chance']
vals   = [others.mean(), naive, grouped, 1/3]
ax.barh(labels[::-1], vals[::-1], color=['#CFC6BF','#1B6470','#8A1C2B','#9A5714'][::-1])
for i, v in enumerate(vals[::-1]):
    ax.text(v + .012, i, f'{v:.1%}', va='center')
ax.set_xlim(0, 1.08); ax.set_xlabel('Accuracy'); ax.grid(axis='x', alpha=.3)
plt.tight_layout(); plt.show()

print('\nNotice what pooling did NOT do: it barely moved the number. The random and grouped')
print('splits also barely differ. Between-person variation is not a splitting problem here —')
print('it is the problem, and no choice of split repairs it.')


---
## Block 7 — One thing that does help: normalise per person

If the problem is that everyone's sensor sits differently and everyone moves at their own amplitude, then remove
exactly that: express every feature as *how far this window is from that person's own average*, in that person's
own standard deviations. The model then sees relative patterns rather than absolute magnitudes.


In [ ]:
Wn = W.copy()
Wn[FEATS] = W.groupby('person')[FEATS].transform(lambda s: (s - s.mean()) / (s.std() + 1e-9))

for label, frame in [('raw features', W), ('per-person normalised', Wn)]:
    sc = cross_val_score(RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1),
                         frame[FEATS], frame.activity,
                         cv=GroupKFold(n_splits=5), groups=frame.person).mean()
    print(f'{label:24s} grouped CV = {sc:.1%}')


> ⚠️ **Do not believe that number.** In this *simulated* data every person differs only by a gain and an offset,
> so per-person standardisation removes the difference exactly and accuracy goes to 100%. Real people differ in
> ways a linear rescaling cannot undo — different gait mechanics, different injuries, different shoes. On real
> recordings expect an improvement of perhaps five to fifteen points, not thirty.
>
> That gap is worth sitting with: **a synthetic dataset can only teach you about the problems its author put into
> it.** The same is true of every simulation study you will read, and of the “validation” in a vendor's whitepaper
> when their test set came from the same twelve athletes as their training set.


---
## Interpretation ✍️ **(graded)**

About 150 words:

1. Why did accuracy drop when you tested on your classmate? Name the specific things the model had learned.
2. What is the equivalent of this problem in an injury model, and in a markerless motion-capture model?
3. You change `WINDOW_S` from 2.0 to 0.5 and accuracy falls. Explain why in terms of the movement, not the maths.
4. A vendor sells you a wearable that classifies training actions at “96% accuracy”. What is the single question you ask?

---

**Your answer:**

*(replace this line)*


---
## AI Use Log

Complete honestly — it is part of the mark. Write *none* if you used no AI tool.

| Tool | What I asked it | What it gave me | What I changed or rejected |
|---|---|---|---|
|  |  |  |  |


---
### Submitting

1. `File → Save a copy in Drive` · 2. `Share → Anyone with the link → Viewer` · 3. Paste the link into ODTÜClass.
